# 🌿 HUẤN LUYỆN MÔ HÌNH NHẬN DIỆN 90 LOÀI KIỂNG LÁ THEO CHUẨN CÁC BÀI BÁO KHOA HỌC QUỐC TẾ
### 📚 Các cải tiến học thuật dựa trên các bài báo mẫu (Reference Papers):
1. **Nội suy Bilinear & Bảo toàn tỷ lệ khung hình (Howard et al., CVPR 2018 - *MobileNetV2*; Tan & Le, ICML 2019 - *EfficientNet*):**
   - Khắc phục lỗi `interpolation='nearest'` ở phiên bản cũ (gây răng cưa gân lá và lệch chuẩn so với trọng số tiền huấn luyện ImageNet vốn dùng `bilinear`).
   - Áp dụng `crop_to_aspect_ratio=True` khi nạp dữ liệu để không làm méo hình học của lá (tránh biến lá dài thành lá tròn).
2. **Tăng cường dữ liệu quang học & không gian ngoài thực địa (PlantCLEF / LifeCLEF; Too et al., 2019 - *Computers and Electronics in Agriculture*):**
   - Bổ sung đầy đủ phép biến đổi thực tế tại vườn: `RandomFlip('horizontal_and_vertical')`, `RandomRotation(0.15)`, `RandomZoom((-0.20, 0.10))`, `RandomTranslation(0.10, 0.10)`, `RandomContrast(0.20)`, và `RandomBrightness(0.20)` giúp mô hình kháng nhiễu ánh nắng gắt, bóng râm và góc chụp nghiêng từ điện thoại.
3. **Label Smoothing ($\varepsilon = 0.1$) + One-Hot Cross-Entropy (Szegedy et al., CVPR 2016; Müller et al., NeurIPS 2019):**
   - Ngăn chặn hiện tượng mạng hội tụ quá tự tin (Overconfidence) vào một số lớp nhiễu nền và giúp phân tách các loài có gân lá tương đồng (họ *Araceae*: *Philodendron*, *Epipremnum*, *Anthurium*, *Aglaonema*).
4. **Hiệu chỉnh trọng số lớp suy giảm mũ (Damped Effective Class Weights - Cui et al., CVPR 2019; Menon et al., ICLR 2021):**
   - Thay vì dùng `class_weight='balanced'` tuyến tính $w_c \propto 1/N_c$ (khiến các loài ít ảnh bị phóng đại trọng số gấp 8–10 lần, dẫn đến đoán nhầm hàng loạt khi gặp nền phức tạp), áp dụng công thức căn bậc hai suy giảm $w_c = (\bar{N}/N_c)^{0.5}$ kết hợp giới hạn $[0.6, 2.2]$.
5. **Chiến lược Fine-Tuning 2 giai đoạn với Cosine Annealing Learning Rate & Mở khóa sâu 60% Backbone + DualPooling (GeM / Avg+Max Pooling):**
   - Kết hợp `GlobalAveragePooling2D` (nắm bắt hình thái tổng thể của tán lá) và `GlobalMaxPooling2D` (nắm bắt chi tiết đốm lá, gân lá sắc nét) theo kiến trúc Fine-Grained Visual Categorization (FGVC).

In [ ]:
# ==============================================================================
# === BƯỚC 1: KẾT NỐI GOOGLE DRIVE & CHUẨN BỊ TẬP DỮ LIỆU 30.000 ẢNH (80-10-10) ===
# ==============================================================================
import os, json, math, random, shutil, zipfile
import numpy as np
import pandas as pd
from pathlib import Path
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from google.colab import drive

if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)
keras.utils.set_random_seed(RANDOM_SEED)

PROJECT = Path('/content/drive/MyDrive/KiengLa_AI')
SAVE_ROOT = PROJECT / 'train_90_lop_30k_paper_standard'
SAVE_ROOT.mkdir(parents=True, exist_ok=True)

RAW_30K_DIR = Path('/content/raw_30k_extracted')
DATA_30K_SPLIT = Path('/content/data_30k_split_80_10_10')

if not (DATA_30K_SPLIT / 'train').exists():
    candidates = [
        PROJECT / '01_Du_Lieu_30k_Va_Labels' / 'merged_v1' / 'merged_90_30k_chuan.zip',
        PROJECT / '01_Du_Lieu_Goc' / 'merged_90_30k_chuan.zip',
        PROJECT / 'merged_v1' / 'merged_90_30k_chuan.zip'
    ]
    zip_path = next((p for p in candidates if p.is_file()), None)
    if zip_path is None:
        found = list(PROJECT.rglob('*merged_90_30k*.zip'))
        if found: zip_path = found[0]
    assert zip_path is not None, 'Không tìm thấy file zip 30k trên Google Drive!'
    print(f'📦 Đang giải nén {zip_path.name}...')
    !unzip -q "{zip_path}" -d /content/raw_30k_extracted

    EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
    class_images = {}
    for root, _, files in os.walk(str(RAW_30K_DIR)):
        folder_name = os.path.basename(root)
        valid_f = [Path(root) / f for f in files if Path(f).suffix.lower() in EXTENSIONS]
        if valid_f and not folder_name.startswith('.'):
            class_images.setdefault(folder_name, []).extend(valid_f)

    class_folders = sorted(list(class_images.keys()))
    assert len(class_folders) == 90, f'Phát hiện {len(class_folders)} lớp (khác 90)!'
    rng = random.Random(RANDOM_SEED)
    for cls_name in class_folders:
        imgs = sorted(list(set(class_images[cls_name])), key=lambda p: p.name)
        rng.shuffle(imgs)
        n = len(imgs)
        n_tr, n_va = int(n * 0.80), int(n * 0.10)
        for split_name, split_imgs in [('train', imgs[:n_tr]), ('val', imgs[n_tr:n_tr+n_va]), ('test', imgs[n_tr+n_va:])]:
            dest = DATA_30K_SPLIT / split_name / cls_name
            dest.mkdir(parents=True, exist_ok=True)
            for p_src in split_imgs:
                try: os.link(p_src, dest / p_src.name)
                except OSError: shutil.copy2(p_src, dest / p_src.name)

print('✅ Tập dữ liệu 30.000 ảnh (80-10-10) đã sẵn sàng!')

In [ ]:
# ==============================================================================
# === BƯỚC 2: DATA PIPELINE CHUẨN BÀI BÁO (BILINEAR + CROP_TO_ASPECT_RATIO)  ===
# ==============================================================================
# 1. Sử dụng interpolation='bilinear' đồng nhất với ImageNet pretraining (Howard et al., 2018)
# 2. Sử dụng crop_to_aspect_ratio=True để không bóp méo hình dạng lá cây
# 3. Sử dụng label_mode='categorical' để hỗ trợ Label Smoothing (Szegedy et al., 2016)

BATCH_SIZE = 32
IMAGE_SIZE = (224, 224)

def load_paper_dataset(split, shuffle):
    return keras.utils.image_dataset_from_directory(
        str(DATA_30K_SPLIT / split),
        image_size=IMAGE_SIZE,
        interpolation='bilinear',
        crop_to_aspect_ratio=True,
        batch_size=BATCH_SIZE,
        label_mode='categorical',
        shuffle=shuffle,
        seed=RANDOM_SEED
    )

train_ds_raw = load_paper_dataset('train', True)
val_ds_raw = load_paper_dataset('val', False)
test_ds_raw = load_paper_dataset('test', False)

class_names = train_ds_raw.class_names
NUM_CLASSES = len(class_names)
assert NUM_CLASSES == 90

# Tính Damped Class Weights theo Cui et al. (CVPR 2019) & Menon et al. (ICLR 2021)
# Tránh việc lớp ít ảnh có trọng số quá lớn gây đoán nhầm (False Positive) ngoài thực tế
y_train_indices = np.concatenate([ np.argmax(y.numpy(), axis=1) for _, y in train_ds_raw ], axis=0)
counts = np.bincount(y_train_indices, minlength=NUM_CLASSES).astype(np.float32)
mean_count = np.mean(counts)
damped_weights = np.clip((mean_count / np.maximum(counts, 1.0)) ** 0.5, 0.65, 2.20)
class_weight_dict = {i: float(w) for i, w in enumerate(damped_weights)}

# Lưu Log-Prior của từng lớp để phục vụ hậu xử lý Logit Adjustment nếu cần
priors = (counts / np.sum(counts)).tolist()
(SAVE_ROOT / 'class_priors.json').write_text(json.dumps({'class_names': class_names, 'counts': counts.tolist(), 'priors': priors}, indent=2, ensure_ascii=False), encoding='utf-8')

train_ds = train_ds_raw.prefetch(tf.data.AUTOTUNE)
val_ds = val_ds_raw.prefetch(tf.data.AUTOTUNE)
test_ds = test_ds_raw.prefetch(tf.data.AUTOTUNE)
print(f'✅ Đã khởi tạo Data Pipeline chuẩn bài báo cho {NUM_CLASSES} loài ({len(y_train_indices):,} ảnh Train).')

In [ ]:
# ==============================================================================
# === BƯỚC 3: XÂY DỰNG KIẾN TRÚC FINE-GRAINED DUAL-POOLING + AUGMENTATION    ===
# ==============================================================================
# Kết hợp GlobalAveragePooling2D + GlobalMaxPooling2D + BatchNormalization (Too et al., 2019)
# Bạn có thể chọn BACKBONE_NAME = 'MobileNetV2' (nhẹ 9.6MB cho Mobile) hoặc 'EfficientNetB0'

BACKBONE_NAME = 'MobileNetV2' # Đổi thành 'EfficientNetB0' nếu muốn train bản 77%+

def build_paper_model(backbone_name='MobileNetV2', num_classes=90):
    inputs = keras.Input(shape=(224, 224, 3), name='image_input')
    
    # Tăng cường dữ liệu thực địa (In-the-wild Nursery Augmentation - PlantCLEF)
    x = layers.RandomFlip('horizontal_and_vertical', seed=RANDOM_SEED)(inputs)
    x = layers.RandomRotation(0.15, seed=RANDOM_SEED)(x)
    x = layers.RandomZoom(height_factor=(-0.20, 0.10), width_factor=(-0.20, 0.10), seed=RANDOM_SEED)(x)
    x = layers.RandomTranslation(0.10, 0.10, seed=RANDOM_SEED)(x)
    x = layers.RandomContrast(0.20, seed=RANDOM_SEED)(x)
    x = layers.RandomBrightness(0.18, value_range=(0, 255), seed=RANDOM_SEED)(x)

    if backbone_name == 'MobileNetV2':
        x = layers.Rescaling(1.0 / 127.5, offset=-1.0, name='rescaling')(x)
        base_model = keras.applications.MobileNetV2(
            input_shape=(224, 224, 3), include_top=False, weights='imagenet'
        )
    else:
        base_model = keras.applications.EfficientNetB0(
            input_shape=(224, 224, 3), include_top=False, weights='imagenet'
        )

    base_model.trainable = False
    feat = base_model(x, training=False)

    # Dual Pooling (Avg + Max) giúp bắt đồng thời hình thái tổng thể & chi tiết đốm/gân lá
    gap = layers.GlobalAveragePooling2D(name='gap')(feat)
    gmp = layers.GlobalMaxPooling2D(name='gmp')(feat)
    pooled = layers.Concatenate(name='dual_pool')([gap, gmp])
    pooled = layers.BatchNormalization(name='head_bn')(pooled)
    pooled = layers.Dropout(0.35, seed=RANDOM_SEED, name='head_drop')(pooled)
    outputs = layers.Dense(num_classes, activation='softmax', name='pred_90')(pooled)

    model = keras.Model(inputs, outputs, name=f'{backbone_name}_FGVC_90')
    return model, base_model

model, base_model = build_paper_model(BACKBONE_NAME, NUM_CLASSES)
model.summary()

In [ ]:
# ==============================================================================
# === BƯỚC 4: HUẤN LUYỆN 2 GIAI ĐOẠN VỚI LABEL SMOOTHING & COSINE ANNEALING  ===
# ==============================================================================
BEST_MODEL_PATH = SAVE_ROOT / f'best_{BACKBONE_NAME.lower()}_paper.keras'

# --- GIAI ĐOẠN 1: WARM-UP CLASSIFICATION HEAD (12 EPOCHS) ---
loss_fn = keras.losses.CategoricalCrossentropy(label_smoothing=0.10)
metrics_list = [
    keras.metrics.CategoricalAccuracy(name='accuracy'),
    keras.metrics.TopKCategoricalAccuracy(k=3, name='top3_acc')
]

model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=1e-3, weight_decay=1e-4),
    loss=loss_fn,
    metrics=metrics_list
)

cb_head = [
    keras.callbacks.ModelCheckpoint(str(BEST_MODEL_PATH), monitor='val_accuracy', mode='max', save_best_only=True, verbose=1),
    keras.callbacks.EarlyStopping(monitor='val_accuracy', mode='max', patience=4, restore_best_weights=True)
]

print('🚀 GIAI ĐOẠN 1: Huấn luyện bộ phân loại Dual-Pooling Head...')
model.fit(train_ds, validation_data=val_ds, epochs=12, class_weight=class_weight_dict, callbacks=cb_head)

# --- GIAI ĐOẠN 2: FINE-TUNING SÂU 80 LỚP CUỐI VỚI COSINE DECAY (25 EPOCHS) ---
base_model.trainable = True
unfreeze_layers = 80 # Mở khóa 80 lớp cuối thay vì chỉ 30 lớp để học sâu đặc trưng gân lá/màu lá
for layer in base_model.layers[:-unfreeze_layers]:
    layer.trainable = False
for layer in base_model.layers[-unfreeze_layers:]:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False # Giữ nguyên thống kê BatchNorm của ImageNet

total_steps = len(train_ds) * 25
cosine_lr = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1e-4, decay_steps=total_steps, alpha=1e-6
)

model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=cosine_lr, weight_decay=1e-4),
    loss=loss_fn,
    metrics=metrics_list
)

cb_fine = [
    keras.callbacks.ModelCheckpoint(str(BEST_MODEL_PATH), monitor='val_accuracy', mode='max', save_best_only=True, verbose=1),
    keras.callbacks.EarlyStopping(monitor='val_accuracy', mode='max', patience=6, restore_best_weights=True),
    keras.callbacks.CSVLogger(str(SAVE_ROOT / f'history_{BACKBONE_NAME.lower()}.csv'))
]

print('🚀 GIAI ĐOẠN 2: Fine-tuning sâu 80 lớp với Cosine Annealing + Label Smoothing...')
model.fit(train_ds, validation_data=val_ds, epochs=25, class_weight=class_weight_dict, callbacks=cb_fine)

# Lưu bản nhẹ không kèm trạng thái Optimizer để nạp nhanh trên Backend local
model.save(str(BEST_MODEL_PATH), include_optimizer=False)
print(f'🎉 Đã lưu model chuẩn bài báo tại: {BEST_MODEL_PATH}')

In [ ]:
# ==============================================================================
# === BƯỚC 5: ĐÁNH GIÁ TRÊN TẬP TEST VỚI MULTI-CROP TTA (PLANTCLEF STANDARD) ===
# ==============================================================================
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

y_true_all, y_probs_all = [], []
for imgs, lbls in test_ds:
    # Trung bình dự đoán giữa ảnh gốc và ảnh lật ngang (Test-Time Augmentation)
    p1 = model(imgs, training=False).numpy()
    p2 = model(tf.image.flip_left_right(imgs), training=False).numpy()
    probs = 0.5 * (p1 + p2)
    y_true_all.append(np.argmax(lbls.numpy(), axis=1))
    y_probs_all.append(probs)

y_true = np.concatenate(y_true_all)
scores = np.concatenate(y_probs_all)
y_pred = np.argmax(scores, axis=1)

top1 = accuracy_score(y_true, y_pred)
top3 = np.mean(np.any(np.argsort(scores, axis=1)[:, -3:] == y_true[:, None], axis=1))
w_f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
print(f'🏆 KẾT QUẢ TEST (CÓ TTA): Top-1 = {top1*100:.2f}% | Top-3 = {top3*100:.2f}% | Weighted F1 = {w_f1*100:.2f}%')